# Tarea 4 — Vbak_Gold_Update

MERGE incremental a `gold_sales_kpis`, agrupando por `sales_org` (VKORG), solo con lo que entró hoy a Silver.
Depende de: `Vbak_Silver_Merge`

In [0]:
CATALOG = "laboratory_sap_dev"
SCHEMA  = "sap_course"
spark.sql(f"USE {CATALOG}.{SCHEMA}")

spark.sql(f'''
    MERGE INTO {CATALOG}.{SCHEMA}.gold_sales_kpis AS g
    USING (
        SELECT
            YEAR(ERDAT)   AS year,
            MONTH(ERDAT)  AS month,
            VKORG         AS sales_org,
            WAERK         AS currency,
            COUNT(*)      AS num_orders_nuevas,
            SUM(NETWR)    AS revenue_nuevo
        FROM {CATALOG}.{SCHEMA}.vbak_silver
        WHERE updated_at = current_date()
        GROUP BY 1, 2, 3, 4
    ) AS n
    ON  g.year = n.year AND g.month = n.month
    AND g.sales_org = n.sales_org AND g.currency = n.currency

    WHEN MATCHED THEN UPDATE SET
        g.num_orders    = g.num_orders + n.num_orders_nuevas,
        g.total_revenue = g.total_revenue + n.revenue_nuevo,
        g.avg_order_value = (g.total_revenue + n.revenue_nuevo)
                             / (g.num_orders + n.num_orders_nuevas)

    WHEN NOT MATCHED THEN INSERT (year, month, sales_org, currency, num_orders, total_revenue, avg_order_value)
    VALUES (n.year, n.month, n.sales_org, n.currency, n.num_orders_nuevas, n.revenue_nuevo,
            n.revenue_nuevo / n.num_orders_nuevas)
''')

print("OK: gold_sales_kpis actualizado con las ordenes de hoy")